# Memory

In [ ]:
!python -m pip install mem0[nlp]

##  1. Setup

In [1]:
from mem0 import Memory

In [2]:
from decouple import Config, RepositoryEnv
import os
env_config = Config(RepositoryEnv("./.env"))

os.environ["GROQ_API_KEY"] = env_config("GROQ_API_KEY")

In [9]:
config = {
    # "history_db_path": "./memory/history.db", # sqlite db path for storing history
    "llm": {
        "provider": "groq",
        "config": {
            "model": "llama-3.3-70b-versatile",
            "temperature": 0.2,
            "max_tokens": 1500,
        },
    },
    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "my_rag_db",
            "path": "./memory/chroma_db",
        },
    },
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "all-MiniLM-L6-v2",
        },
    },
}

In [10]:
memory = Memory.from_config(config)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4781.55it/s]
/home/niuniu/Documents/tmp/test/.venv/lib64/python3.14/site-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


## 2. Manage Data

In [11]:
memory.add([{"role": "user", "content": "Hello I am John. I am interested in learning Mem0 and LangGraph."}], user_id="user1")
memory.add([{"role": "user", "content": "I just learned about how to set up the memory.using custom llm, vector store and embedder."}], user_id="user1", metadata={"topic": "setup"})
memory.add([{"role": "user", "content": "I went to the store this afternoon to get some bread."}, {"role": "ai", "content": "What kind of bread did you get?"}], user_id="user1", metadata={"topic": "private"})

Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


{'results': [{'id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
   'memory': 'User went to the store in the afternoon of June 9, 2026, to get some bread',
   'event': 'ADD'}]}

In [29]:
# get all memories for user1
memories = memory.get_all(filters={"user_id": "user1"})
memories

{'results': [{'id': '18caccff-9a77-4f2d-8d43-2aa86ee57704',
   'memory': "User's name is John and he is interested in learning Mem0 and LangGraph as of June 9, 2026",
   'hash': 'edc100b76247f9dfc17b312f3fb5a6e7',
   'metadata': None,
   'created_at': '2026-06-09T20:10:04.187740+00:00',
   'updated_at': '2026-06-09T20:10:04.187740+00:00',
   'user_id': 'user1'},
  {'id': '3729d0ad-1b8d-40c9-b2da-b56b2549ee1e',
   'memory': 'User learned about setting up the memory using custom LLM, vector store, and embedder as of June 9, 2026',
   'hash': 'c447540b207897b0b2cedc67bb307123',
   'metadata': {'topic': 'setup'},
   'created_at': '2026-06-09T20:10:30.826386+00:00',
   'updated_at': '2026-06-09T20:10:30.826386+00:00',
   'user_id': 'user1'},
  {'id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
   'memory': "Actually I didn't go to the store, I just bought bread online.",
   'hash': '30c3d49d3029327088f8da4a6aade994',
   'metadata': {'topic': 'private'},
   'created_at': '2026-06-09T20:11:09.913

In [17]:
# filter the metadata
related = memory.search(
    "Where did I buy?", 
    filters={
        "user_id": "user1"
    }, 
    top_k=2)
related 

{'results': [{'id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
   'memory': 'User went to the store in the afternoon of June 9, 2026, to get some bread',
   'hash': '7dbc58ee6a74e8f91be3cf57bd9ca742',
   'metadata': {'topic': 'private'},
   'score': 1.0,
   'created_at': '2026-06-09T20:11:09.913766+00:00',
   'updated_at': '2026-06-09T20:11:09.913766+00:00',
   'user_id': 'user1'},
  {'id': '3729d0ad-1b8d-40c9-b2da-b56b2549ee1e',
   'memory': 'User learned about setting up the memory using custom LLM, vector store, and embedder as of June 9, 2026',
   'hash': 'c447540b207897b0b2cedc67bb307123',
   'metadata': {'topic': 'setup'},
   'score': 1.0,
   'created_at': '2026-06-09T20:10:30.826386+00:00',
   'updated_at': '2026-06-09T20:10:30.826386+00:00',
   'user_id': 'user1'}]}

In [13]:
# filter the metadata
related = memory.search(
    "Where did I buy?", 
    filters={
        "user_id": "user1", 
        "topic": "setup"
    })
related 

{'results': [{'id': '3729d0ad-1b8d-40c9-b2da-b56b2549ee1e',
   'memory': 'User learned about setting up the memory using custom LLM, vector store, and embedder as of June 9, 2026',
   'hash': 'c447540b207897b0b2cedc67bb307123',
   'metadata': {'topic': 'setup'},
   'score': 1.0,
   'created_at': '2026-06-09T20:10:30.826386+00:00',
   'updated_at': '2026-06-09T20:10:30.826386+00:00',
   'user_id': 'user1'}]}

In [14]:
# get a single memory by id
data = memory.get("3729d0ad-1b8d-40c9-b2da-b56b2549ee1e")
data

{'id': '3729d0ad-1b8d-40c9-b2da-b56b2549ee1e',
 'memory': 'User learned about setting up the memory using custom LLM, vector store, and embedder as of June 9, 2026',
 'hash': 'c447540b207897b0b2cedc67bb307123',
 'metadata': {'topic': 'setup'},
 'score': None,
 'created_at': '2026-06-09T20:10:30.826386+00:00',
 'updated_at': '2026-06-09T20:10:30.826386+00:00',
 'user_id': 'user1'}

In [31]:
# update memory
memory.update(
    memory_id="904ba67c-c00e-47dd-90d1-79ef7b9fc740", 
    data="User went to the store in the afternoon of June 9, 2026, to get some bread."
)
data = memory.get("904ba67c-c00e-47dd-90d1-79ef7b9fc740")
data

{'id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
 'memory': 'User went to the store in the afternoon of June 9, 2026, to get some bread.',
 'hash': 'da09fcc27c01f6ca5ea69b8037881fef',
 'metadata': {'topic': 'private'},
 'score': None,
 'created_at': '2026-06-09T20:11:09.913766+00:00',
 'updated_at': '2026-06-09T20:31:36.228954+00:00',
 'user_id': 'user1'}

In [68]:
type(data)

dict

In [62]:
# display change history of a memory
history = memory.history(memory_id="904ba67c-c00e-47dd-90d1-79ef7b9fc740")
history

[{'id': 'e1f9227f-d179-4159-b982-92b53e1f227a',
  'memory_id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
  'old_memory': None,
  'new_memory': 'User went to the store in the afternoon of June 9, 2026, to get some bread',
  'event': 'ADD',
  'created_at': '2026-06-09T20:11:09.913766+00:00',
  'updated_at': None,
  'is_deleted': False,
  'actor_id': None,
  'role': None},
 {'id': 'eb956f3f-0ca5-46ca-8326-27821a1fce68',
  'memory_id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
  'old_memory': 'User went to the store in the afternoon of June 9, 2026, to get some bread',
  'new_memory': "Actually I didn't go to the store, I just bought bread online.",
  'event': 'UPDATE',
  'created_at': '2026-06-09T20:11:09.913766+00:00',
  'updated_at': '2026-06-09T20:29:20.966256+00:00',
  'is_deleted': False,
  'actor_id': None,
  'role': None},
 {'id': 'c325daff-ef24-4953-ad4a-15de2afefe54',
  'memory_id': '904ba67c-c00e-47dd-90d1-79ef7b9fc740',
  'old_memory': "Actually I didn't go to the store, I just bou

## 3. LangGraph Integration

In [51]:
# State
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_id: str

In [52]:
# llm
from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

In [ ]:
# chatbot node
from langchain_core.messages import SystemMessage, HumanMessage

def chatbot_node(state: State) -> State:

    # 1. get the mem0 instance
    global memory

    # 2. get state
    messages = state["messages"]
    user_id = state["user_id"]

    # 3. retrieve related memories of the current user
    related_memories = memory.search(
        query=messages[-1].content,
        filters={"user_id": user_id},
        top_k=5
    )["results"]
    
    # 4. construct the context with retrieved memories
    if related_memories:
        context = "\n\n".join([f"Memory: {m['memory']}" for m in related_memories])
    else:
        context = "No previous relevant information."

    system_prompt = f"""You are a helpful assistant. Here is some relevant information from the user's memory: {context}"""

    # 5. generate response with llm
    response = llm.invoke([SystemMessage(content=system_prompt)] + messages)

    # 6. save the conversation to memory
    memory.add([
        {"role": "user", "content": messages[-1].content},
        {"role": "assistant", "content": response.content},
    ], user_id=user_id)

    return {"messages": response}

In [64]:
# build graph
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)
graph.add_node("chatbot", chatbot_node)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

agent = graph.compile()

In [67]:
# tests
while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    state = {"messages": [HumanMessage(content=user_input)], "user_id": "user1"}
    final_state = agent.invoke(state)
    print(f"Assistant: {final_state['messages'][-1].content}")

DEBUG:  [HumanMessage(content='Do you know who I am', additional_kwargs={}, response_metadata={}, id='25383b4f-5267-46ff-acf2-ca44d0561e50')]
Assistant: I'm a bit confused. Initially, I interacted with a user named John who was interested in learning about Mem0 and LangGraph. However, later you introduced yourself as Rose. I'm not sure if you're the same person or a different individual. Could you please clarify?
DEBUG:  [HumanMessage(content='', additional_kwargs={}, response_metadata={}, id='99487812-4a4b-4dbe-ad06-ba69ca7fe8bd')]


KeyboardInterrupt: 